# Notebook 2b: Sensitivity Check (run on Google Colab with a GPU)

The main comparison keeps the settings from the proposal fixed for both models (dropout 0.2, learning rate 0.001)
and only searches the latent size. This notebook checks whether the conclusions depend on those fixed choices.
It is run on IEEE-CIS only, because BAF shows almost no signal for either model.

| Variant | Changed setting | Models |
|---|---|---|
| Main (from Notebook 2) | none | AE, VAE |
| No dropout | dropout 0.0 | AE, VAE |
| Lower learning rate | learning rate 0.0003 | AE, VAE |
| Weaker KL | beta 0.001 | VAE |
| Stronger KL | beta 0.1 | VAE |

Each variant is trained with seeds 0, 1 and 2 (18 runs, roughly 30 minutes on a T4 GPU) and compared with
seeds 0, 1 and 2 of the main runs. Evaluation uses the same rules as Notebook 3: the calibrated threshold
comes from the validation period and the metrics from the test period.

Before running on Colab: upload the `04_Final_Thesis` folder (with `Processed/`, `Results/` and `Final_Code/`)
to `My Drive`, and switch on the GPU (Runtime, Change runtime type, T4 GPU).

**At a glance**

| | |
|---|---|
| Run on | Google Colab with a GPU |
| Run time | About 30 minutes on a T4 GPU (18 training runs) |
| Reads | `../Processed/ieee.npz`, `../Results/runs_final/`, `best_latent.json`, `thesis_models.py` |
| Writes | `../Results/runs_sensitivity/`, `tables/table_sensitivity.csv`, `tables/sensitivity_all_runs.csv` |
| Run order | Step 3 of 6, after Notebook 02. |

In [1]:
import os, sys, json, time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/04_Final_Thesis"
    sys.path.insert(0, os.path.join(BASE, "Final_Code"))
else:
    BASE = os.path.abspath("..")
    sys.path.insert(0, os.getcwd())
import thesis_models as tm
print("Running on:", tm.DEVICE)

PROC = os.path.join(BASE, "Processed")
MAIN = os.path.join(BASE, "Results", "runs_final")
SENS = os.path.join(BASE, "Results", "runs_sensitivity")
os.makedirs(SENS, exist_ok=True); os.makedirs(os.path.join(BASE, "Results", "tables"), exist_ok=True)
with open(os.path.join(BASE, "Results", "best_latent.json")) as f:
    BEST = json.load(f)
HIDDEN, BETA, SEEDS = (256, 128, 64), 0.01, [0, 1, 2]

d = np.load(os.path.join(PROC, "ieee.npz"))
data = {k: d[k] for k in ["X_train", "X_es", "X_val", "y_val", "X_test", "y_test"]}
print({k: v.shape for k, v in data.items()})

Mounted at /content/drive
Running on: cuda
{'X_train': (358788, 419), 'X_es': (40052, 419), 'X_val': (88581, 419), 'y_val': (88581,), 'X_test': (88581, 419), 'y_test': (88581,)}


In [2]:
VARIANTS = [
    ("AE",  "no dropout",          "_drop0",  dict(dropout=0.0)),
    ("AE",  "learning rate 0.0003", "_lr3e-4", dict(lr=3e-4)),
    ("VAE", "no dropout",          "_drop0",  dict(dropout=0.0, beta=BETA)),
    ("VAE", "learning rate 0.0003", "_lr3e-4", dict(lr=3e-4, beta=BETA)),
    ("VAE", "beta 0.001",          "",        dict(beta=0.001)),
    ("VAE", "beta 0.1",            "",        dict(beta=0.1)),
]
t_all = time.time()
for kind, label, tag, kw in VARIANTS:
    for s in SEEDS:
        tm.run_experiment("ieee", kind, data, HIDDEN, BEST[f"ieee_{kind}"], seed=s, out_root=SENS,
                          tag=tag, verbose=False, **kw)
print(f"Finished in {(time.time()-t_all)/60:.0f} min")

ieee_AE_z32_s0_drop0: training ...
ieee_AE_z32_s0_drop0: done in 80s, 150 epochs, val AUC-PR 0.1759, val AUC-ROC 0.7456
ieee_AE_z32_s1_drop0: training ...
ieee_AE_z32_s1_drop0: done in 79s, 150 epochs, val AUC-PR 0.1702, val AUC-ROC 0.7441
ieee_AE_z32_s2_drop0: training ...
ieee_AE_z32_s2_drop0: done in 78s, 150 epochs, val AUC-PR 0.1687, val AUC-ROC 0.7449
ieee_AE_z32_s0_lr3e-4: training ...
ieee_AE_z32_s0_lr3e-4: done in 87s, 150 epochs, val AUC-PR 0.1649, val AUC-ROC 0.7255
ieee_AE_z32_s1_lr3e-4: training ...
ieee_AE_z32_s1_lr3e-4: done in 95s, 150 epochs, val AUC-PR 0.1600, val AUC-ROC 0.7222
ieee_AE_z32_s2_lr3e-4: training ...
ieee_AE_z32_s2_lr3e-4: done in 86s, 150 epochs, val AUC-PR 0.1643, val AUC-ROC 0.7285
ieee_VAE_b0.01_z16_s0_drop0: training ...
ieee_VAE_b0.01_z16_s0_drop0: done in 105s, 150 epochs, val AUC-PR 0.1625, val AUC-ROC 0.7131
ieee_VAE_b0.01_z16_s1_drop0: training ...
ieee_VAE_b0.01_z16_s1_drop0: done in 103s, 150 epochs, val AUC-PR 0.1620, val AUC-ROC 0.7159
ieee

In [3]:
def calibrated_f1(s_val, s_test):
    p, r, t = precision_recall_curve(data["y_val"], s_val)
    f1 = 2 * p * r / np.clip(p + r, 1e-12, None)
    thr = t[np.argmax(f1[:-1])]
    return f1_score(data["y_test"], (s_test >= thr).astype(int))

def evaluate(path):
    meta = json.load(open(os.path.join(path, "meta.json")))
    sv, st = (np.load(os.path.join(path, f"scores_{k}.npy")) for k in ("val", "test"))
    return {"auc_roc": roc_auc_score(data["y_test"], st), "auc_pr": average_precision_score(data["y_test"], st),
            "f1": calibrated_f1(sv, st), "epochs": meta["epochs_run"], "active_units": meta.get("active_latent_units")}

rows = []
for kind in ["AE", "VAE"]:
    z = BEST[f"ieee_{kind}"]
    for s in SEEDS:
        name = f"ieee_AE_z{z}_s{s}" if kind == "AE" else f"ieee_VAE_b{BETA:g}_z{z}_s{s}"
        rows.append({"model": kind, "variant": "main (dropout 0.2, lr 0.001, beta 0.01)", "seed": s,
                     **evaluate(os.path.join(MAIN, name))})
for kind, label, tag, kw in VARIANTS:
    z = BEST[f"ieee_{kind}"]
    for s in SEEDS:
        b = kw.get("beta", BETA)
        name = (f"ieee_AE_z{z}_s{s}" if kind == "AE" else f"ieee_VAE_b{b:g}_z{z}_s{s}") + tag
        rows.append({"model": kind, "variant": label, "seed": s, **evaluate(os.path.join(SENS, name))})

sens = pd.DataFrame(rows)
tab = sens.groupby(["model", "variant"], sort=False)[["auc_roc", "auc_pr", "f1", "epochs", "active_units"]].agg(["mean", "std"]).round(4)
tab.to_csv(os.path.join(BASE, "Results", "tables", "table_sensitivity.csv"))
sens.to_csv(os.path.join(BASE, "Results", "tables", "sensitivity_all_runs.csv"), index=False)
tab

auc_roc          auc_pr          \
                                                 mean     std    mean     std   
model variant                                                                   
AE    main (dropout 0.2, lr 0.001, beta 0.01)  0.7100  0.0046  0.0880  0.0012   
VAE   main (dropout 0.2, lr 0.001, beta 0.01)  0.7051  0.0031  0.0853  0.0009   
AE    no dropout                               0.7375  0.0027  0.0979  0.0026   
      learning rate 0.0003                     0.7164  0.0038  0.0890  0.0017   
VAE   no dropout                               0.6987  0.0074  0.0852  0.0023   
      learning rate 0.0003                     0.7008  0.0038  0.0838  0.0005   
      beta 0.001                               0.7116  0.0053  0.0885  0.0017   
      beta 0.1                                 0.6431  0.0080  0.0652  0.0013   

                                                   f1            epochs  \
                                                 mean     std      mean   
model variant                                                             
AE    main (dropout 0.2, lr 0.001, beta 0.01)  0.1821  0.0018  150.0000   
VAE   main (dropout 0.2, lr 0.001, beta 0.01)  0.1742  0.0051  129.3333   
AE    no dropout                               0.1979  0.0040  150.0000   
      learning rate 0.0003                     0.1820  0.0037  150.0000   
VAE   no dropout                               0.1809  0.0046  142.6667   
      learning rate 0.0003                     0.1741  0.0030  150.0000   
      beta 0.001                               0.1832  0.0034  150.0000   
      beta 0.1                                 0.1367  0.0013   96.3333   

                                                       active_units          
                                                   std         mean     std  
model variant                                                                
AE    main (dropout 0.2, lr 0.001, beta 0.01)   0.0000          NaN     NaN  
VAE   main (dropout 0.2, lr 0.001, beta 0.01)  26.8390       5.3333  0.5774  
AE    no dropout                                0.0000          NaN     NaN  
      learning rate 0.0003                      0.0000          NaN     NaN  
VAE   no dropout                               12.7017       6.6667  0.5774  
      learning rate 0.0003                      0.0000       5.3333  0.5774  
      beta 0.001                                0.0000      13.3333  1.1547  
      beta 0.1                                 20.5994       2.6667  0.5774

**How to read this:** if the AE stays ahead of the VAE under every variant, and no variant moves either model
close to Isolation Forest (AUC-ROC 0.771), then the main conclusions do not depend on the fixed settings.